In [ ]:
%load_ext autoreload

In [ ]:
import sys
from pathlib import Path
import tomllib

import altair as alt
import geopandas as gpd
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# HOTFIX because we don't have a central repository for travel diary survey scripts yet
sys.path.append("../2022/04b-summary_notebooks/")
%autoreload
from altair_utils import color_value_transit_teal, color_sfcore_restofsf_restofbayarea, color_covid_pre_post_both
from downtown_travel_study_utils import load_hh_2018, load_hh_2022, parse_geog
from utils import (
    load_person_assign_day,
    load_trip_assign_day,
    trip_join_hh_person,
)

In [ ]:
# this pipeline just follows precedent from the 2022/04b-summary_notebooks:
# this may not be the best way to process the data, but until I manage to fully review
# the 2019-tnc pipeline (which was copied over basically verbatim to make the 2022
# pipeline), I'll just stick with whatever we did for the 2022 pipeline)

In [ ]:
with open("../2022_v03/pipeline_config.toml", "rb") as f:
    config_2022 = tomllib.load(f)
with open("config-2018.toml", "rb") as f:
    config_2018 = tomllib.load(f)

In [ ]:
dir_2018 = Path(
    "Q:/Data/Surveys/HouseholdSurveys/MTC-SFCTA2018/Processing_20211018/in_2022_format"
)
dir_2022 = Path("Q:/Data/Surveys/HouseholdSurveys/MTC-SFCTA2022/Processed_20241127\v03")

In [ ]:
taz_analysis_neighborhoods_gis_filepath = r"Q:\GIS\Policy\San_Francisco\Analysis_Neighborhoods\taz2454-sf_only-with_analysis_neighborhoods.gpkg"
taz_analysis_neighborhoods = gpd.read_file(taz_analysis_neighborhoods_gis_filepath)[
    ["TAZ", "analysis_neighborhood"]
]

In [ ]:
def load_trip(config, year):
    hh_taz_spatial_join_filepath = Path(config["01-taz_spatial_join"]["dir"]) / "hh.csv"
    if year == 2018:
        hh = load_hh_2018(hh_taz_spatial_join_filepath).drop("hhtaz").join(
            pl.read_csv(Path(config["02a-reformat"]["dir"]) / "hh.csv").select(["hhno", "hhtaz"]),
            on=["hhno"],
            how="left"
        )
        assign_day_dir = Path(config["03b-assign_day"]["dir"]) / "wt_wkday"
    elif year == 2022:
        hh = load_hh_2022(hh_taz_spatial_join_filepath)
        assign_day_dir = Path(config["03b-assign_day"]["dir"]) / "wt-wkday_3day"
    trip_assign_day_filepath = (assign_day_dir / "trip.csv")
    person = load_person_assign_day(
        assign_day_dir / "person.csv",
        Path(config["00-preprocess"]["dir"]) / "person.csv",
        Path(config["02a-reformat"]["dir"]) / "person.csv",
        demographics=True,
        year=year,
    )
    trip = parse_geog(
        trip_join_hh_person(
            load_trip_assign_day(trip_assign_day_filepath),  # depart_hour=False
            hh,
            person,
        ),
        taz_analysis_neighborhoods,
    ).with_columns(
        pl.col("mode").replace_strict(
            {
                1: "walk/bike",  # "1. walk",
                2: "walk/bike",  # "2. bike",
                3: "automobile",  # "3. sov",
                4: "automobile",  # "4. hov2",
                5: "automobile",  # "5. hov3+",
                6: "transit / school bus",  # "6. walk-transit",
                7: "transit / school bus",  # "7. drive-transit",
                8: "transit / school bus",  # "8. school bus",
                9: "taxi/TNC",  # "9. Taxi/TNC",
            }
        ),).with_columns(
        dpurp=pl.col("dpurp").replace_strict(
            {
                0: "other",  # "00 none/home",
                1: "work",
                2: "school/escort", # "02 school",
                3: "school/escort", # "03 escort",
                4: "shop / pers bus / med", # "04 pers bus + med",
                5: "shop / pers bus / med", # "05 shop",
                6: "meal/social/rec",  # "06 meal",
                7: "meal/social/rec",  # "07 social+rec",
                11: "other",  # "11 overnight non home + other"
                # 10: "10",
                -1: "-1"
            }
        ),
        o_in_ne_core=(pl.col("o_geog") == "1. northeast core"),
        d_in_ne_core=(pl.col("d_geog") == "1. northeast core"),
    ).with_columns(
        o_and_d_in_ne_core=(pl.col("o_in_ne_core") & pl.col("d_in_ne_core")),
        o_or_d_in_ne_core=(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")))
    if year == 2018:
        # parse income to 2022 categories
        trip = trip.with_columns(pl.col("hh_income_cat").replace({
            "000-25k": "000-100k",
            "025-50k": "000-100k",
            "050-75k": "000-100k",
            "075-100k": "000-100k",
            "100-150k": "100-200k",
            "150-200k": "100-200k",
            "200-250k": "200k+",
            "250k+": "200k+",
        }))
    elif year == 2022:
        # drop non adults, since 2018 dataset does NOT include adults
        # the categories around 18 are 16-17 then 18-24
        trip = trip.filter(pl.col("pagey") >= 18).with_columns(
            pl.col("hh_income_cat").replace({
                "000-25k": "000-100k",
                "025-50k": "000-100k",
                "050-75k": "000-100k",
                "075-100k": "000-100k",
            })
        )
    return trip


def trips_to_persons(trip):
    return (
        trip.group_by(
            "hhno",
            "pno",
            # demographics from person table
            "pagey",
            "pgend",
            "raceeth",
            # income from hh (2022) or person (2018) table
            "hh_income_cat",
            # home location is derived from hh table
            "hh_geog",
            "home_county",
            # the weights should be the same for each person/hh
            # (depending on how RSG propagated weights from hh to person)
            # since for the 2018 batch, weighting was only done at the hh level
            "psexpfac"  # person weights
        )
        .agg(  # "ne" = "northeast"
            any_trip_o_in_ne_core=(pl.col("o_geog") == "1. northeast core").any(),
            any_trip_d_in_ne_core=(pl.col("d_geog") == "1. northeast core").any(),
            # trip purpose:
            ne_core_opurps=pl.when(pl.col("o_geog") == "1. northeast core")
            .then(pl.col("opurp"))
            .otherwise(pl.lit(None))
            .unique()
            .drop_nulls(),
            ne_core_dpurps=pl.when(pl.col("d_geog") == "1. northeast core")
            .then(pl.col("dpurp"))
            .otherwise(pl.lit(None))
            .unique()
            .drop_nulls(),
        )
        .with_columns(
            ne_core_purps=pl.col("ne_core_opurps")
            .list.concat(pl.col("ne_core_dpurps"))
            .list.unique()
        )
    )


def plot_dist(df, df_name, col, x_label_angle=0):
    df = (
        df.group_by(col).agg(weighted=pl.sum("trexpfac"), unweighted=pl.len()).sort(col)
    )
    return df, alt.Chart(df).mark_bar().encode(
        x=alt.X(f"{col}:O", title=None, axis=alt.Axis(labelAngle=x_label_angle)), y=alt.Y("weighted", title=col), color=color_value_transit_teal
    ).properties(title=f"{df_name}")
    # + df.plot.bar(
    #     x=col, y="unweighted", title=f"{df_name}\n{col}: unweighted", frame_width=200
    # ).opts(xrotation=45)


def add_share_col(df):
    return df.with_columns(
        (pl.col("weighted") / pl.col("weighted").sum().over("year")).alias(
            f"weighted-share"
        )
    )


def plot_dists(df2018, df2022, df_name, col, x_label_angle=0):
    df2018, plot2018 = plot_dist(df2018, f"{df_name} preCOVID", col, x_label_angle=x_label_angle)
    df2022, plot2022 = plot_dist(df2022, f"{df_name} postCOVID", col, x_label_angle=x_label_angle)
    df = add_share_col(pl.concat(
        [
        df2018.with_columns(year=pl.lit(2019)),
        df2022.with_columns(year=pl.lit(2023)),
        ]
    ))
    display(df)
    return df, (plot2018 | plot2022).resolve_scale(y='shared')


def plot_heatmap(trip, rows_var, cols_var, title, normalized_to_col=True):
    # unweighted trips heatmap
    sns.heatmap(
        trip
        .sort(rows_var, cols_var)
        .pivot(
            index=rows_var,
            on=cols_var,
            values="trexpfac",
            aggregate_function="len",
        )
        .to_pandas()
        .set_index(rows_var),
        annot=True,
        annot_kws={"size": 7},
    )
    plt.title(title + ": entries")
    plt.show()

    if normalized_to_col:
        trip = trip.with_columns(
            trexpfac_normalized_to_cols=(
                pl.col("trexpfac") / pl.col("trexpfac").sum().over(cols_var)
            )
        )
        values = "trexpfac_normalized_to_cols"
    else:
        values = "trexpfac"
    trip_pivoted = trip.sort(rows_var, cols_var).pivot(
        index=rows_var,
        on=cols_var,
        values="trexpfac",
        aggregate_function="sum",
    )
    trip_pivoted_shares = trip.sort(rows_var, cols_var).pivot(
        index=rows_var,
        on=cols_var,
        values="trexpfac_normalized_to_cols",
        aggregate_function="sum",
    )
    sns.heatmap(
        trip.sort(rows_var, cols_var).pivot(
            index=rows_var,
            on=cols_var,
            values=values,
            aggregate_function="sum",
        )
        .to_pandas()
        .set_index(rows_var),
        vmax=0.7,
        annot=True,
        annot_kws={"size": 7},
    )
    plt.title(title)
    plt.show()
    return trip_pivoted, trip_pivoted_shares

In [ ]:
trip_2018 = load_trip(config_2018, 2018)
trip_2022 = load_trip(config_2022, 2022)

In [ ]:
person_2018 = trips_to_persons(trip_2018)
person_2022 = trips_to_persons(trip_2022)

In [ ]:
print("total trips in 2018:", trip_2018.select(pl.sum("trexpfac")).item())
print("total trips in 2022:", trip_2022.select(pl.sum("trexpfac")).item())

In [ ]:
print("total persons in each hh home geog")
person_2018.group_by("hh_geog").agg(pl.len().alias("2018-unweighted"), pl.sum("psexpfac").alias("2018-weighted")).join(
    person_2022.group_by("hh_geog").agg(pl.len().alias("2022-unweighted"), pl.sum("psexpfac").alias("2022-weighted")), "hh_geog").sort("hh_geog")

In [ ]:
def od_by_geog(trip):
    # rows: O \ columns: D
    return trip.group_by("o_geog", "d_geog").agg(pl.sum("trexpfac")).pivot(
        index="o_geog", on="d_geog", values="trexpfac", sort_columns=True
    ).sort("o_geog").rename({"o_geog": "trip geog: O\\D"})

od_by_geog_2018 = od_by_geog(trip_2018)
od_by_geog_2022 = od_by_geog(trip_2022)

print(2018)
display(od_by_geog_2018)
print(2022)
display(od_by_geog_2022)

# Joe thinks that the rest of Bay Area -> rest of Bay Area results are suspiciously high

od_by_geog_2018.write_csv("output/od_by_geog_2019.csv")
od_by_geog_2022.write_csv("output/od_by_geog_2023.csv")

In [ ]:
# _, chart = plot_dists(trip_2018.filter(pl.col("o_in_ne_core")),
#     trip_2022.filter(pl.col("o_in_ne_core")), "trip (O in NE core)", "hh_geog")
# chart

In [ ]:
trip_d_in_ne_core_hh_geog_df, _ = plot_dists(trip_2018.filter(pl.col("d_in_ne_core")), trip_2022.filter(pl.col("d_in_ne_core")), "trip (D in NE core)", "hh_geog",
x_label_angle=-45)
trip_d_in_ne_core_hh_geog_df = trip_d_in_ne_core_hh_geog_df.with_columns(
    pl.col("hh_geog").replace({"1. northeast core": "NE core", "2. rest of SF": "rest of SF", "3. rest of Bay Area"	: "rest of Bay Area"	}),
)
trip_d_in_ne_core_hh_geog_df.write_csv("output/trip-d_in_ne_core-hh_geog.csv")

In [ ]:
trip_d_in_ne_core_hh_geog_chart = (
    alt.Chart(trip_d_in_ne_core_hh_geog_df)
    .mark_line()
    .encode(
        x=alt.X("year:O", scale=alt.Scale(padding=3)),
        y=alt.Y("weighted", title="trips to/within NE core by home location"),
        color=alt.Color("hh_geog").scale(
            domain=["NE core", "rest of SF", "rest of Bay Area"],
            range=color_sfcore_restofsf_restofbayarea,
        ),
        # tooltip=tooltip
    )
).interactive()
trip_d_in_ne_core_hh_geog_chart

In [ ]:
# (
#     alt.Chart(trip_d_in_ne_core_hh_geog_df)
#     .mark_line()
#     .encode(
#         x=alt.X("year:O", scale=alt.Scale(padding=3)),
#         y=alt.Y(
#             "weighted-share", title="share of trips to/within NE core by home location"
#         ),
#         color=alt.Color("hh_geog").scale(
#             domain=["NE core", "rest of SF", "rest of Bay Area"],
#             range=color_sfcore_restofsf_restofbayarea,
#         ),
#         # tooltip=tooltip
#     )
# ).interactive()

In [ ]:
# trip_d_in_restofsf_hh_geog_df, _ = plot_dists(trip_2018.filter((pl.col("d_geog") == "2. rest of SF")), trip_2022.filter((pl.col("d_geog") == "2. rest of SF")), "trip (D in rest of SF)", "hh_geog",
# x_label_angle=-45)
# trip_d_in_restofsf_hh_geog_df = trip_d_in_restofsf_hh_geog_df.with_columns(
#     pl.col("hh_geog").replace({"1. northeast core": "NE core", "2. rest of SF": "rest of SF", "3. rest of Bay Area"	: "rest of Bay Area"	}),
# )
# trip_d_in_restofsf_hh_geog_df.write_csv("output/trip-d_in_restofsf-hh_geog.csv")

In [ ]:
# (
#     alt.Chart(trip_d_in_restofsf_hh_geog_df)
#     .mark_line()
#     .encode(
#         x=alt.X("year:O", scale=alt.Scale(padding=3)),
#         y=alt.Y("weighted", title="trips to/within rest of SF by home location"),
#         color=alt.Color("hh_geog").scale(
#             domain=["NE core", "rest of SF", "rest of Bay Area"],
#             range=color_sfcore_restofsf_restofbayarea,
#         ),
#         # tooltip=tooltip
#     )
# ).interactive()

In [ ]:
# (
#     alt.Chart(trip_d_in_restofsf_hh_geog_df)
#     .mark_line()
#     .encode(
#         x=alt.X("year:O", scale=alt.Scale(padding=3)),
#         y=alt.Y(
#             "weighted-share",
#             title="share of trips to/within rest of SF by home location",
#         ),
#         color=alt.Color("hh_geog").scale(
#             domain=["NE core", "rest of SF", "rest of Bay Area"],
#             range=color_sfcore_restofsf_restofbayarea,
#         ),
#         # tooltip=tooltip
#     )
# ).interactive()

In [ ]:
# TODO make map of hh_geog by analysis neighborhood for trip (D in NE core)

In [ ]:
# trip_d_in_ne_core_purpose_df, _ = plot_dists(trip_2018.filter(pl.col("d_in_ne_core")), trip_2022.filter(pl.col("d_in_ne_core")), "trip (D in NE core)", "dpurp",
# x_label_angle=-45)
# trip_d_in_ne_core_purpose_df.write_csv("output/trip-d_in_ne_core-purpose.csv")

In [ ]:
# trip_d_in_ne_core_purpose_chart = (
#     alt.Chart(trip_d_in_ne_core_purpose_df)
#     .mark_line()
#     .encode(
#         x=alt.X("year:O", scale=alt.Scale(padding=3)),
#         y=alt.Y("weighted", title="trips to/within NE core by purpose"),
#         color="dpurp",
#         # alt.Color("hh_geog").scale(
#         #     domain=[
#         #         "NE core",
#         #         "rest of SF",
#         #         "rest of Bay Area",
#         #     ],
#         #     range=color_sfcore_restofsf_restofbayarea,
#         # ),
#         # tooltip=tooltip
#     )
# ).interactive()
# trip_d_in_ne_core_purpose_chart

In [ ]:
# imputed gender
# _, chart = plot_dists(trip_2018.filter(pl.col("d_in_ne_core")).with_columns(pl.col("gender_imputed").replace_strict({1: "female", 2:"male"})), trip_2022.filter(pl.col("d_in_ne_core")), "trip (D in NE core)", "gender_imputed")
# chart

In [ ]:
# pgend
# _, chart = plot_dists(trip_2018.filter(pl.col("d_in_ne_core")), trip_2022.filter(pl.col("d_in_ne_core")), "trip (D in NE core)", "pgend")
# chart

In [ ]:
# trip_d_in_ne_core_raceeth_df, _ = plot_dists(trip_2018.filter(pl.col("d_in_ne_core")), trip_2022.filter(pl.col("d_in_ne_core")), "trip (D in NE core)", "raceeth",
# x_label_angle=-45)
# trip_d_in_ne_core_raceeth_df.write_csv("output/trip-d_in_ne_core-raceeth.csv")

In [ ]:
# trip_d_in_ne_core_raceeth_chart = (
#     alt.Chart(trip_d_in_ne_core_raceeth_df)
#     .mark_line()
#     .encode(
#         x=alt.X("year:O", scale=alt.Scale(padding=3)),
#         y=alt.Y("weighted", title="trips to/within NE core by race/ethnicity"),
#         color="raceeth",
#         # alt.Color("hh_geog").scale(
#         #     domain=[
#         #         "NE core",
#         #         "rest of SF",
#         #         "rest of Bay Area",
#         #     ],
#         #     range=color_sfcore_restofsf_restofbayarea,
#         # ),
#         # tooltip=tooltip
#     )
# ).interactive()
# trip_d_in_ne_core_raceeth_chart

In [ ]:
# trip_d_in_ne_core_income_df, _ = plot_dists(trip_2018.filter(pl.col("d_in_ne_core")), trip_2022.filter(pl.col("d_in_ne_core")), "trip (D in NE core)", "hh_income_cat",
# x_label_angle=-45)
# trip_d_in_ne_core_income_df.write_csv("output/trip-d_in_ne_core-income.csv")

In [ ]:
# trip_d_in_ne_core_income_chart = (
#     alt.Chart(trip_d_in_ne_core_income_df)
#     .mark_line()
#     .encode(
#         x=alt.X("year:O", scale=alt.Scale(padding=3)),
#         y=alt.Y("weighted", title="trips to/within NE core by household income"),
#         color="hh_income_cat",
#         # alt.Color("hh_geog").scale(
#         #     domain=[
#         #         "NE core",
#         #         "rest of SF",
#         #         "rest of Bay Area",
#         #     ],
#         #     range=color_sfcore_restofsf_restofbayarea,
#         # ),
#         # tooltip=tooltip
#     )
# ).interactive()
# trip_d_in_ne_core_income_chart

In [ ]:
# TODO dpurp shares by hour of day

In [ ]:
trips_od_in_necore_2018 = trip_2018.filter(
    pl.col("o_in_ne_core") & pl.col("d_in_ne_core")
)
trips_o_outside_d_within_ne_core_2018 = trip_2018.filter(
    ~pl.col("o_in_ne_core") & pl.col("d_in_ne_core")
)
trips_o_restofsf_d_necore_2018 = trip_2018.filter(
    ((pl.col("o_geog") == "2. rest of SF") & pl.col("d_in_ne_core"))
)
trips_o_restofbayarea_d_necore_2018 = trip_2018.filter(
    (pl.col("o_geog") == "3. rest of Bay Area") & pl.col("d_in_ne_core")
)
trips_tofromwithin_necore_2018 = trip_2018.filter(
    pl.col("o_in_ne_core") | pl.col("d_in_ne_core")
)
# trips_cross_ne_core_boundary_2018 = trip_2018.filter(
#     ((pl.col("o_geog") == "northeast core") & (pl.col("d_geog") != "northeast core"))
#     | ((pl.col("o_geog") != "northeast core") & (pl.col("d_geog") == "northeast core"))
# )

trips_od_in_necore_2022 = trip_2022.filter(
    pl.col("o_in_ne_core") & pl.col("d_in_ne_core")
)
trips_o_outside_d_within_ne_core_2022 = trip_2022.filter(
    ~pl.col("o_in_ne_core") & pl.col("d_in_ne_core")
)
trips_o_restofsf_d_necore_2022 = trip_2022.filter(
    ((pl.col("o_geog") == "2. rest of SF") & pl.col("d_in_ne_core"))
)
trips_o_restofbayarea_d_necore_2022 = trip_2022.filter(
    (pl.col("o_geog") == "3. rest of Bay Area") & pl.col("d_in_ne_core")
)
trips_tofromwithin_necore_2022 = trip_2022.filter(
    pl.col("o_in_ne_core") | pl.col("d_in_ne_core")
)

In [ ]:
trips_tofromwithin_necore_2018.group_by("mode").agg(pl.sum("trexpfac")).sort("mode")

In [ ]:
trips_tofromwithin_necore_2022.group_by("mode").agg(pl.sum("trexpfac")).sort("mode")

In [ ]:
trips_tofromwithin_necore_2022.select(pl.sum("trexpfac"))

In [ ]:
trip_od_in_necore_mode_df, _ = plot_dists(
    trips_od_in_necore_2018, trips_od_in_necore_2022,
    "trips o&d within NE core",
    "mode",
    x_label_angle=-45
)
trip_od_in_necore_mode_df.write_csv("output/trip-od_in_necore-mode.csv")

In [ ]:
trip_od_in_necore_mode_chart = (
    alt.Chart(trip_od_in_necore_mode_df)
    .mark_line()
    .encode(
        x=alt.X("year:O", scale=alt.Scale(padding=3)),
        y=alt.Y("weighted", title="intra-northeast core trips by mode"),
        color="mode",
        # alt.Color("hh_geog").scale(
        #     domain=[
        #         "NE core",
        #         "rest of SF",
        #         "rest of Bay Area",
        #     ],
        #     range=color_sfcore_restofsf_restofbayarea,
        # ),
        # tooltip=tooltip
    )
).interactive()
trip_od_in_necore_mode_chart

In [ ]:
trip_o_restofsf_d_necore_mode_df, _ = plot_dists(
    trips_o_restofsf_d_necore_2018,
    trips_o_restofsf_d_necore_2022,
    "trips o rest of SF & d NE core",
    "mode",
    x_label_angle=-45
)
trip_o_restofsf_d_necore_mode_df.write_csv("output/trip-o_restofsf_d_necore-mode.csv")

In [ ]:
trip_o_restofsf_d_necore_mode_chart = (
    alt.Chart(trip_o_restofsf_d_necore_mode_df)
    .mark_line()
    .encode(
        x=alt.X("year:O", scale=alt.Scale(padding=3)),
        y=alt.Y("weighted", title="trips from rest of SF to northeast core by mode"),
        color="mode",
        # alt.Color("hh_geog").scale(
        #     domain=[
        #         "NE core",
        #         "rest of SF",
        #         "rest of Bay Area",
        #     ],
        #     range=color_sfcore_restofsf_restofbayarea,
        # ),
        # tooltip=tooltip
    )
).interactive()
trip_o_restofsf_d_necore_mode_chart

In [ ]:
trip_o_restofbayarea_d_necore_mode_df, _ = plot_dists(
    trips_o_restofbayarea_d_necore_2018,
    trips_o_restofbayarea_d_necore_2022,
    "trips o rest of Bay Area & d NE core",
    "mode",
    x_label_angle=-45
)
trip_o_restofbayarea_d_necore_mode_df.write_csv("output/trip-o_restofbayarea_d_necore-mode.csv")

In [ ]:
trip_o_restofbayarea_d_necore_mode_chart = (
    alt.Chart(trip_o_restofbayarea_d_necore_mode_df)
    .mark_line()
    .encode(
        x=alt.X("year:O", scale=alt.Scale(padding=3)),
        y=alt.Y("weighted", title="trips from rest of Bay Area to northeast core by mode"),
        color="mode",
        # alt.Color("hh_geog").scale(
        #     domain=[
        #         "NE core",
        #         "rest of SF",
        #         "rest of Bay Area",
        #     ],
        #     range=color_sfcore_restofsf_restofbayarea,
        # ),
        # tooltip=tooltip
    )
).interactive()
trip_o_restofbayarea_d_necore_mode_chart

In [ ]:
# # now let's just look at work trips and their hh geog
# # (though work trips may be different from commute trips)
# plot_dists(
#     trip_2018.filter(pl.col("d_in_ne_core") & (pl.col("dpurp") == "work")), trip_2022.filter(pl.col("d_in_ne_core") & (pl.col("dpurp") == "work")),
#     "trips: d within NE core, dpurp == work (1)",
#     "hh_geog",
# )

In [ ]:
def crosstab_1year(trip, group1, group2, shares=True):
    """
    shares are of group1 over group2
    """
    df = trip.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")).group_by(group1, group2).agg(unweighted=pl.len(), weighted=pl.sum("trexpfac")
    )
    if shares:
        df = df.with_columns((pl.col("weighted") / pl.sum("weighted").over(group2)).alias(f"{group1}-shares"))
    return df.sort(group1, group2)

def crosstab_compare(trip_2018, trip_2022, group1, group2):
    return pl.concat(
        [crosstab_1year(trip_2018, group1, group2).with_columns(year=pl.lit(2019)), crosstab_1year(trip_2022, group1, group2).with_columns(year=pl.lit(2023))]
    )

In [ ]:
mode_x_income_df = crosstab_compare(trip_2018, trip_2022, "mode", "hh_income_cat")
mode_x_income_df.write_csv("output/trip-tofromwithin_necore-mode-income.csv")
alt.Chart(mode_x_income_df).mark_bar().encode(
    x="year:O",
    y="weighted",
    color="mode",
    column="hh_income_cat",
)

In [ ]:
mode_x_income_df.group_by("hh_income_cat", "year").agg(
    pl.sum("unweighted",	"weighted",	"mode-shares")).with_columns(mode=pl.lit("total")).select(
        "mode",	"hh_income_cat",	"unweighted",	"weighted",	"mode-shares",	"year"
    )

In [ ]:
alt.Chart(mode_x_income_df).mark_bar().encode(
    x="year:O",
    y="mode-shares",
    color="mode",
    column="hh_income_cat",
)

In [ ]:
mode_x_dpurp_df = crosstab_compare(trip_2018, trip_2022, "mode", "dpurp")
mode_x_dpurp_df.write_csv("output/trip-tofromwithin_necore-mode-dpurp.csv")
alt.Chart(mode_x_dpurp_df).mark_bar().encode(
    x="year:O",
    y="weighted",
    color="mode",
    column="dpurp",
)

In [ ]:
alt.Chart(mode_x_dpurp_df).mark_bar().encode(
    x="year:O",
    y="mode-shares",
    color="mode",
    column="dpurp",
)

In [ ]:
mode_x_raceeth_df = crosstab_compare(trip_2018, trip_2022, "mode", "raceeth")
mode_x_raceeth_df.write_csv("output/trip-tofromwithin_necore-mode-raceeth.csv")
alt.Chart(mode_x_raceeth_df).mark_bar().encode(
    x="year:O",
    y="weighted",
    color="mode",
    column="raceeth",
)

In [ ]:
alt.Chart(mode_x_raceeth_df).mark_bar().encode(
    x="year:O",
    y="mode-shares",
    color="mode",
    column="raceeth",
)

In [ ]:
hhgeog_x_dpurp_df = crosstab_compare(trip_2018, trip_2022, "hh_geog", "dpurp")
hhgeog_x_dpurp_df.write_csv("output/trip-tofromwithin_necore-hh_geog-dpurp.csv")
alt.Chart(hhgeog_x_dpurp_df).mark_bar().encode(
    x="year:O",
    y="weighted",
    color="hh_geog",
    column="dpurp",
)

In [ ]:
alt.Chart(hhgeog_x_dpurp_df).mark_bar().encode(
    x="year:O",
    y="hh_geog-shares",
    color="hh_geog",
    column="dpurp",
)

In [ ]:
trip_tofromwithin_necore_mode_income_2018, trip_tofromwithin_necore_mode_income_shares_2018 = plot_heatmap(
    trip_2018.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")), "mode", "hh_income_cat", "trip 2018 tofromwithin_necore mode x income\neach col sum normalized to 1",
    normalized_to_col=True)
trip_tofromwithin_necore_mode_income_2018.write_csv("output/trip-tofromwithin_necore-mode-income-2018.csv")
trip_tofromwithin_necore_mode_income_shares_2018.write_csv("output/trip-tofromwithin_necore-mode-income-shares-2018.csv")

In [ ]:
trip_tofromwithin_necore_mode_income_2022, trip_tofromwithin_necore_mode_income_shares_2022 = plot_heatmap(
    trip_2022.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")), "mode", "hh_income_cat",
    "trip 2022 tofromwithin_necore mode x income\neach col sum normalized to 1", normalized_to_col=True)
trip_tofromwithin_necore_mode_income_2022.write_csv("output/trip-tofromwithin_necore-mode-income-2022.csv")
trip_tofromwithin_necore_mode_income_shares_2022.write_csv("output/trip-tofromwithin_necore-mode-income-shares-2022.csv")

In [ ]:
plot_heatmap(trip_2018.filter(pl.col("d_in_ne_core")), "mode", "pagey", "trip 2018 d_in_ne_core mode x age\neach col sum normalized to 1", normalized_to_col=True)

In [ ]:
plot_heatmap(trip_2022.filter(pl.col("d_in_ne_core")), "mode", "pagey", "trip 2022 d_in_ne_core mode x age\neach col sum normalized to 1", normalized_to_col=True)

In [ ]:
trip_tofromwithin_necore_mode_raceeth_2018, trip_tofromwithin_necore_mode_raceeth_shares_2018 = plot_heatmap(trip_2018.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")),
    "mode", "raceeth", "trip 2018 tofromwithin_necore mode x raceeth\neach col sum normalized to 1", normalized_to_col=True)
trip_tofromwithin_necore_mode_raceeth_2018.write_csv("output/trip-tofromwithin_necore-mode-raceeth-2018.csv")
trip_tofromwithin_necore_mode_raceeth_shares_2018.write_csv("output/trip-tofromwithin_necore-mode-raceeth-shares2018.csv")

In [ ]:
trip_tofromwithin_necore_mode_raceeth_2022, trip_tofromwithin_necore_mode_raceeth_shares_2022 = plot_heatmap(trip_2022.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")),
    "mode", "raceeth", "trip 2022 tofromwithin_necore mode x raceeth\neach col sum normalized to 1", normalized_to_col=True)
trip_tofromwithin_necore_mode_raceeth_2022.write_csv("output/trip-tofromwithin_necore-mode-raceeth-2022.csv")
trip_tofromwithin_necore_mode_raceeth_shares_2022.write_csv("output/trip-tofromwithin_necore-mode-raceeth-shares-2022.csv")

In [ ]:
trip_tofromwithin_necore_mode_dpurp_2018, trip_tofromwithin_necore_mode_dpurp_shares_2018 = plot_heatmap(trip_2018.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")),
    "mode", "dpurp", "trip 2018 d_in_ne_core mode x dpurp\neach col sum normalized to 1", normalized_to_col=True)
trip_tofromwithin_necore_mode_dpurp_2018.write_csv("output/trip-tofromwithin_necore-mode-dpurp-2018.csv")
trip_tofromwithin_necore_mode_dpurp_shares_2018.write_csv("output/trip-tofromwithin_necore-mode-dpurp-shares-2018.csv")

In [ ]:
trip_tofromwithin_necore_mode_dpurp_2022, trip_tofromwithin_necore_mode_dpurp_shares_2022 = plot_heatmap(trip_2022.filter(pl.col("o_in_ne_core") | pl.col("d_in_ne_core")),
    "mode", "dpurp", "trip 2022 d_in_ne_core mode x dpurp\neach col sum normalized to 1", normalized_to_col=True)
trip_tofromwithin_necore_mode_dpurp_2022.write_csv("output/trip-tofromwithin_necore-mode-dpurp-2022.csv")
trip_tofromwithin_necore_mode_dpurp_shares_2022.write_csv("output/trip-tofromwithin_necore-mode-dpurp-shares-2022.csv")

In [ ]:
pl.concat([
    trip_tofromwithin_necore_mode_dpurp_2018.with_columns(year=2018),
    trip_tofromwithin_necore_mode_dpurp_2022.with_columns(year=2022)
])

In [ ]:
alt.Chart(pl.concat(
    [
        trip_2018.filter(pl.col("o_and_d_in_ne_core")).group_by(depart_hour=(pl.col("deptm") // 100)).agg(pl.sum("trexpfac")).sort("depart_hour").with_columns(year=2018),
        trip_2022.filter(pl.col("o_and_d_in_ne_core")).group_by(depart_hour=(pl.col("deptm") // 100)).agg(pl.sum("trexpfac")).sort("depart_hour").with_columns(year=2022),
    ]
)).mark_line().encode(
    x="depart_hour",
    y="trexpfac",
    color="year:N"
).properties(title="trips dept hour, o_and_d_in_ne_core, 2018v2022")

In [ ]:
alt.Chart(pl.concat(
    [
        trip_2018.filter(pl.col("o_or_d_in_ne_core")).group_by(depart_hour=(pl.col("deptm") // 100)).agg(pl.sum("trexpfac")).sort("depart_hour").with_columns(year=2018),
        trip_2022.filter(pl.col("o_or_d_in_ne_core")).group_by(depart_hour=(pl.col("deptm") // 100)).agg(pl.sum("trexpfac")).sort("depart_hour").with_columns(year=2022),
    ]
)).mark_line().encode(
    x="depart_hour",
    y="trexpfac",
    color="year:N"
).properties(title="trips dept hour, o_or_d_in_ne_core, 2018v2022")

In [ ]:
alt.Chart(pl.concat(
    [
        trip_2018.filter(True).group_by(depart_hour=(pl.col("deptm") // 100)).agg(pl.sum("trexpfac")).sort("depart_hour").with_columns(year=2018),
        trip_2022.filter(True).group_by(depart_hour=(pl.col("deptm") // 100)).agg(pl.sum("trexpfac")).sort("depart_hour").with_columns(year=2022),
    ]
)).mark_line().encode(
    x="depart_hour",
    y="trexpfac",
    color="year:N"
).properties(title="trips dept hour, entire Bay, 2018v2022")

In [ ]:
# IGNORE ALL PERSON BASED ANALYSIS BELOW

In [ ]:
persons_2018 = (
    trip_2018.group_by(
        "hhno",
        "pno",
        # demographics from person table
        "pagey",
        "pgend",
        "raceeth",
        # income from hh table
        # "hh_income_cat",  # TODO
        # home location is derived from hh table
        "hh_geog",
        "home_county",
        # the weights should be the same for each person/hh
        # (depending on how RSG propagated weights from hh to person)
        # since for the 2018 batch, weighting was only done at the hh level
    )
    .agg(  # "ne" = "northeast"
        pl.sum("trexpfac"),
        any_trip_o_in_ne_core=(pl.col("o_geog") == "northeast core").any(),
        any_trip_d_in_ne_core=(pl.col("d_geog") == "northeast core").any(),
        # trip purpose:
        ne_core_opurps=pl.when(pl.col("o_geog") == "northeast core")
        .then(pl.col("opurp"))
        .otherwise(pl.lit(None))
        .unique()
        .drop_nulls(),
        ne_core_dpurps=pl.when(pl.col("d_geog") == "northeast core")
        .then(pl.col("dpurp"))
        .otherwise(pl.lit(None))
        .unique()
        .drop_nulls(),
    )
    .with_columns(
        ne_core_purps=pl.col("ne_core_opurps")
        .list.concat(pl.col("ne_core_dpurps"))
        .list.unique()
    )
)

In [ ]:
# sanity check for the number of persons in survey dataset
# population: SF ~ 8e5, Bay Area ~ 7.7e6

In [ ]:
persons_2018.group_by("hh_geog").agg(pl.sum("trexpfac"))

In [ ]:
persons_2018.group_by("home_county").agg(pl.sum("trexpfac")).sort("home_county")

In [ ]:
print("persons with orig/dest of any trip in the northeast core:")
persons_2018.group_by("any_trip_o_in_ne_core").agg(
    o_weighted=pl.sum("trexpfac"), o_unweighted=pl.len()
).rename({"any_trip_o_in_ne_core": "any_trip_in_ne_core"}).join(
    persons_2018.group_by("any_trip_d_in_ne_core")
    .agg(d_weighted=pl.sum("trexpfac"), d_unweighted=pl.len())
    .rename({"any_trip_d_in_ne_core": "any_trip_in_ne_core"}),
    on="any_trip_in_ne_core",
)
# TODO expand this to the 3 geographies

In [ ]:
# persons: persons with orig/dest of any trip in the northeast core
o_or_d_in_ne_core_persons_2018 = persons_2018.filter(
    pl.col("any_trip_o_in_ne_core") | pl.col("any_trip_d_in_ne_core")
)

In [ ]:
# TODO QUESTION!: can we even group by days if the weights are for the 4 day period

In [ ]:
plot_dist(o_or_d_in_ne_core_persons_2018, "persons in NE core", "pgend")

In [ ]:
plot_dist(o_or_d_in_ne_core_persons_2018, "persons in NE core", "pagey")

In [ ]:
plot_dist(o_or_d_in_ne_core_persons_2018, "persons in NE core", "raceeth")

In [ ]:
plot_dist(o_or_d_in_ne_core_persons_2018, "persons in NE core", "hh_geog")

In [ ]:
plot_dist(o_or_d_in_ne_core_persons_2018, "persons in NE core", "home_county")

In [ ]:
plot_dist(o_or_d_in_ne_core_persons_2018, "persons in NE core", "hh_income_cat")

In [ ]:
# purpose: how many purposes? work only vs non-work only vs mix? ...?
plot_dist(
    o_or_d_in_ne_core_persons_2018.with_columns(
        ne_core_purp_grouped=pl.when(pl.col("ne_core_purps") == [0])
        .then(pl.lit("0: home only"))
        .when((pl.col("ne_core_purps") == [1]) | (pl.col("ne_core_purps") == [0, 1]))
        .then(pl.lit("1 (+ 0): work (+ home) only"))
        .when((pl.col("ne_core_purps") == [2]) | (pl.col("ne_core_purps") == [0, 2]))
        .then(pl.lit("2: school (+ home) only"))
        .when(pl.col("ne_core_purps").list.contains(1))
        .then(pl.lit("work + other purp"))
        .otherwise(pl.lit("other"))
    ),
    "persons in NE core",
    "ne_core_purp_grouped",
)